# BFCL-India v3: Fine-Tune ToolCaller-Qwen-3B

**Goal:** Fine-tune Qwen2.5-3B-Instruct with QLoRA on 9.4K Indian-context function-calling examples.

**Setup:**
- **Platform:** Kaggle T4 (free, 30hr/week) or Colab T4
- **Model:** Qwen/Qwen2.5-3B-Instruct (4-bit QLoRA)
- **Dataset:** bhavjeetsingh2912/toolcaller-train-mix (8,443 train + 938 val)
- **Training:** 3 epochs, ~2-3 hours on T4

**Before running:**
1. Enable GPU: Settings > Accelerator > GPU T4 x2 (Kaggle) or Runtime > T4 (Colab)
2. Add your HF_TOKEN as a Kaggle Secret or Colab Secret (needed for model upload)

## 1. Install Dependencies

In [ ]:
!pip install -q --no-warn-conflicts \
    torch \
    transformers>=4.51.0 \
    datasets>=2.21.0 \
    accelerate>=1.0.0 \
    peft>=0.13.0 \
    trl>=0.12.0 \
    bitsandbytes>=0.43.0 \
    huggingface-hub>=0.30.0

!nvidia-smi

## 2. Authenticate with Hugging Face

Set your `HF_TOKEN` as a Kaggle/Colab secret, or paste it below.

In [ ]:
import os

# Try Kaggle secrets first
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle secrets')
except Exception:
    pass

# Try Colab secrets
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
        print('HF_TOKEN loaded from Colab secrets')
    except Exception:
        pass

if 'HF_TOKEN' not in os.environ:
    print('WARNING: HF_TOKEN not found. Model upload will fail.')
    print("Set it as a Kaggle/Colab secret, or run: os.environ['HF_TOKEN'] = 'hf_...'")
else:
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('Logged in to Hugging Face Hub')

## 3. Load Model (4-bit QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
MAX_SEQ_LEN = 2048

# 4-bit quantization config - critical for fitting on T4 (16GB VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False

print(f'Model loaded: {MODEL_NAME}')
print(f'GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB')

## 4. Attach LoRA Adapter

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,          # 2x rank - standard for QLoRA
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_dropout=0.05,      # slight regularization for 9K examples
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Load Dataset

Downloads from HuggingFace Hub - no need to upload data files manually.

In [ ]:
from datasets import load_dataset

# Load from HuggingFace Hub
dataset = load_dataset(
    'bhavjeetsingh2912/toolcaller-train-mix',
    data_files={'train': 'train.jsonl', 'val': 'val.jsonl'},
    verification_mode='no_checks',
)

print(f"Train: {len(dataset['train'])} examples")
print(f"Val:   {len(dataset['val'])} examples")

# Check source distribution
from collections import Counter
src_counts = Counter(dataset['train']['source'])
print('\nTraining mix:')
for src, cnt in src_counts.most_common():
    print(f"  {src}: {cnt} ({cnt/len(dataset['train'])*100:.1f}%)")

## 6. Format for SFT

Apply Qwen's chat template to convert `messages` arrays into the text format the model expects.

In [ ]:
def format_for_sft(examples):
    texts = []
    for messages in examples['messages']:
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {'text': texts}

dataset = dataset.map(format_for_sft, batched=True, num_proc=2)

# Check token lengths to verify no truncation issues
sample_lengths = [
    len(tokenizer.encode(t)) for t in dataset['train']['text'][:500]
]
import statistics
print(f'Token length stats (first 500):')
print(f'  Median: {statistics.median(sample_lengths):.0f}')
print(f'  Mean:   {statistics.mean(sample_lengths):.0f}')
print(f'  Max:    {max(sample_lengths)}')
print(f'  >2048:  {sum(1 for l in sample_lengths if l > MAX_SEQ_LEN)} / {len(sample_lengths)}')

# Preview a sample
print('\n' + '='*60)
print('SAMPLE (first 500 chars):')
print('='*60)
print(dataset['train'][0]['text'][:500])

## 7. Train

- **3 epochs** on ~8.4K examples = ~3,150 steps (batch=2, grad_accum=4)
- Eval every 200 steps, save best checkpoint by eval_loss
- Gradient checkpointing for memory efficiency
- Expected time: ~2-3 hours on T4

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = 'outputs_v3'

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    # Batch size: effective = 2 * 4 = 8
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    # Schedule
    num_train_epochs=3,
    warmup_steps=100,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    # Precision and memory
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    # Logging and eval
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=200,
    eval_accumulation_steps=4,
    # Checkpointing
    save_strategy='steps',
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    # Optimizer
    optim='paged_adamw_8bit',
    # Sequence length
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field='text',
    packing=False,
    # Misc
    seed=3407,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['val'],
    args=training_args,
)

print(f"Total training steps: ~{len(dataset['train']) * 3 // 8}")
print(f'Eval every {training_args.eval_steps} steps')
print('Training...')

trainer_stats = trainer.train()
print('\n' + '='*60)
print('TRAINING COMPLETE')
print('='*60)
print(f"Total time: {trainer_stats.metrics['train_runtime']/3600:.1f} hours")
print(f"Final train loss: {trainer_stats.metrics['train_loss']:.4f}")

## 8. Loss Curves

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_loss = [(x['step'], x['loss']) for x in log_history if 'loss' in x]
eval_loss = [(x['step'], x['eval_loss']) for x in log_history if 'eval_loss' in x]

if train_loss and eval_loss:
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot([s for s, _ in train_loss], [l for _, l in train_loss],
            label='Train Loss', color='#4A90D9', linewidth=2)
    ax.plot([s for s, _ in eval_loss], [l for _, l in eval_loss],
            label='Eval Loss', color='#E74C3C', linewidth=2, marker='o', markersize=6)
    ax.set_xlabel('Step', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title('BFCL-India v3: Training Progress', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('loss_curve_v3.png', dpi=150, bbox_inches='tight')
    plt.show()

    final_train = train_loss[-1][1]
    best_eval = min(l for _, l in eval_loss)
    final_eval = eval_loss[-1][1]
    gap = final_eval - final_train
    print(f'Final train loss:  {final_train:.4f}')
    print(f'Best eval loss:    {best_eval:.4f}')
    print(f'Final eval loss:   {final_eval:.4f}')
    print(f'Train-eval gap:    {gap:.4f}')
    if gap > 0.3:
        print('WARNING: Large gap suggests overfitting. Consider fewer epochs.')
    elif gap > 0.15:
        print('Slight overfitting. Results should still be good.')
    else:
        print('Healthy generalization.')
else:
    print('No loss data recorded - check training logs.')

## 9. Save LoRA Adapter Locally

In [ ]:
ADAPTER_DIR = 'v3_lora_adapter'

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'LoRA adapter saved to {ADAPTER_DIR}/')

# Show adapter size
import os
total_size = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if os.path.isfile(os.path.join(ADAPTER_DIR, f))
)
print(f'Adapter size: {total_size / 1024**2:.1f} MB')

## 10. Merge and Upload to Hugging Face

Merges the LoRA weights back into the base model and pushes the full model to HF Hub.

In [ ]:
HF_REPO = 'bhavjeetsingh2912/toolcaller-qwen-3b-v3'

# Merge LoRA into base model
print('Merging LoRA adapter into base model...')
merged_model = model.merge_and_unload()

# Push to Hub
print(f'Pushing to {HF_REPO}...')
merged_model.push_to_hub(
    HF_REPO,
    commit_message='v3: QLoRA fine-tune on 9.4K examples (60% Indian context)',
    private=False,
)
tokenizer.push_to_hub(
    HF_REPO,
    commit_message='v3: tokenizer',
)
print(f'\nModel uploaded to https://huggingface.co/{HF_REPO}')

## 11. Sanity Test: Quick Inference

Run a few Indian-context queries through the model to verify it outputs valid JSON.

In [ ]:
import json# Use the merged model for inferencemerged_model.config.use_cache = Truemerged_model.eval()SYSTEM_PROMPT = """You are a tool-calling assistant. The user's request must be answered ONLY by calling one or more of the tools provided.Today's date is 2026-06-15. When the user says "tomorrow", "next Friday", "in 3 days", resolve to an absolute YYYY-MM-DD date against this anchor and put the resolved date in the tool args. Never put words like "tomorrow" inside args.Output a single JSON object: {"calls": [{"tool": "<name>", "args": {...}}, ...]}.Rules:- If multiple tools are needed (parallel), include all of them in the "calls" array.- For multi-turn conversations, output ONLY the next call(s) given the conversation so far.- If NO available tool can satisfy the request, output {"calls": []}.- Do not invent tool names. Do not invent argument keys not in the schema.- Honour all regex patterns, enums, and required fields.- Output strict JSON. No markdown, no prose, no explanation.AVAILABLE TOOLS:upi_send(recipient_vpa: string*, amount: number*, currency: INR*, note: string) - Send money via UPI to a VPA. Use when user wants to SEND or PAY money.irctc_search_trains(from_station: string*, to_station: string*, date: string*, travel_class: SL|3A|2A|1A|CC|EC*, quota: GN|TQ|LD|PT) - Search trains on IRCTC. Use when user asks to FIND or SEARCH trains.swiggy_search_restaurant(query: string*, city: string*, cuisine: string, veg_only: boolean) - Search restaurants on Swiggy by name, cuisine, or dish."""test_queries = [    "paaji ko 500 rupees bhej de upi pe, unka id paaji@okaxis",    "Delhi se Mumbai ke liye kal ka Rajdhani dikhao, 2A class mein",    "Bangalore mein koi accha South Indian restaurant dhundho Swiggy pe, veg only",]print("=" * 70)print("SANITY TEST: Model Inference")print("=" * 70)for i, query in enumerate(test_queries, 1):    messages = [        {"role": "system", "content": SYSTEM_PROMPT},        {"role": "user", "content": query},    ]    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)    inputs = tokenizer(text, return_tensors="pt").to(merged_model.device)    with torch.no_grad():        output_ids = merged_model.generate(            **inputs, max_new_tokens=256, do_sample=False, temperature=1.0,        )    generated = tokenizer.decode(        output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True,    ).strip()    try:        parsed = json.loads(generated)        valid = "calls" in parsed        status = "VALID" if valid else "Missing 'calls' key"    except json.JSONDecodeError:        status = "INVALID JSON"        parsed = None    print(f"\n--- Test {i} [{status}] ---")    print(f"Query: {query}")    print(f"Output: {generated[:300]}")    if parsed:        print(f"Parsed: {json.dumps(parsed, indent=2, ensure_ascii=False)[:300]}")print("\n" + "=" * 70)print("Done! If all 3 tests show VALID, the model is ready for full eval.")print("=" * 70)

## 12. Create Model Card

In [ ]:
MODEL_CARD = """---license: mitbase_model: Qwen/Qwen2.5-3B-Instructdatasets:  - bhavjeetsingh2912/toolcaller-train-mixtags:  - function-calling  - tool-use  - indian-context  - qlora  - bfcllanguage:  - en  - hipipeline_tag: text-generation---# ToolCaller-Qwen-3B v3**Qwen2.5-3B-Instruct fine-tuned for Indian-context function calling.**Trained on 9,381 examples (60% Indian-context, 20% xLAM, 20% Glaive+APIGen-MT+seeds)using QLoRA (r=16, alpha=32) for 3 epochs on a Kaggle T4 GPU.## Use CaseCall Indian APIs: UPI payments, IRCTC train booking, Aadhaar verification, Swiggy orders,and 46 more tools using natural language in English, Hindi, or Hinglish.## Training Data| Source | Count | Pct ||---|---|---|| Indian-context (BFCL-India) | 6,150 | 65.6% || xLAM unfiltered | 2,050 | 21.9% || APIGen-MT | 820 | 8.7% || Glaive | 347 | 3.7% || BFCL seeds | 10 | 0.1% || xLAM filtered | 4 | <0.1% |## Output FormatThe model outputs strict JSON:{"calls": [{"tool": "upi_send", "args": {"recipient_vpa": "user@upi", "amount": 500, "currency": "INR"}}]}## Benchmark: BFCL-India| Model | Params | Weighted Accuracy ||---|---|---|| Gemini 2.5 Flash | ? | 75.9% || Llama 3.3 70B | 70B | 74.3% || GPT-4o-mini | ? | 69.1% || **ToolCaller-Qwen-3B v3** | **3B** | **TBD** |## Important: Date AnchoringThe system prompt MUST include today's date. Relative date expressions ("tomorrow","next Friday") are resolved against this anchor. Using a stale date produces wrongabsolute dates silently.## Links- Benchmark and eval: [github.com/bhavjeetsingh/bfcl-India](https://github.com/bhavjeetsingh/bfcl-India)- Training data: [bhavjeetsingh2912/toolcaller-train-mix](https://huggingface.co/datasets/bhavjeetsingh2912/toolcaller-train-mix)- License: MIT"""from huggingface_hub import HfApiapi = HfApi()api.upload_file(    path_or_fileobj=MODEL_CARD.encode(),    path_in_repo="README.md",    repo_id=HF_REPO,    repo_type="model",    commit_message="Add model card",)print(f"Model card uploaded to https://huggingface.co/{HF_REPO}")

## Next Steps

1. **Run full eval:** `python eval.py --model bhavjeetsingh2912/toolcaller-qwen-3b-v3 --provider hf --device cuda`
2. **Compare against baselines** (GPT-4o-mini: 69.1%, Llama-70B: 74.3%, Gemini: 75.9%)
3. **Update README leaderboard** with your model's score
4. **Deploy Gradio demo** on HuggingFace Spaces